# FLIR Proxy YOLO Split Analysis

This notebook compares the `train`, `val`, and `test` splits of the FLIR proxy YOLO dataset used by the experiment configs. The goal is to make the validation/evaluation gap concrete by looking at:

- single object instances: classes, box sizes, positions, and difficult singleton slices;
- slice distributions: joint `(class_label, size_bin, position_bin)` support using the same 3x3 position bins and global size tertiles as `docs/notebooks/flir_slice_subset_study.ipynb`;
- image-level properties: empty images, instance density, class density, small-object share, and optional pixel statistics.

The notebook is intentionally split into reusable cells. Run the setup and loading cells once, then rerun the analysis blocks independently while changing `YOLO_DATASET_YAML`, `SPLIT_ORDER`, or the plotting parameters.

In [ ]:
from __future__ import annotations

import json
import math
import sys
from collections.abc import Iterable
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import Markdown, display
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / 'src').exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT))

from src.analysis.flir_subgroup.yolo_slice_stats import (  # noqa: E402
    POSITION_BIN_ORDER,
    SIZE_BIN_ORDER,
    add_position_bin_columns,
    assign_bins_from_thresholds,
    build_slice_counts_table,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

YOLO_DATASET_YAML = REPO_ROOT / 'data' / 'derived' / 'yolo-test-ds' / 'full_train.yaml'
EXPORT_MANIFEST = REPO_ROOT / 'data' / 'derived' / 'yolo-test-ds' / 'export_manifest.json'
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'analysis' / 'yolo_split_analysis'
SPLIT_ORDER = ['train', 'val', 'test']
PIXEL_STATS_SAMPLE_PER_SPLIT = 500
RANDOM_SEED = 1337

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO_ROOT}')
print(f'Dataset YAML: {YOLO_DATASET_YAML}')
print(f'Output directory: {OUTPUT_DIR}')

## Dataset Discovery

This cell resolves image and label directories from the YOLO dataset YAML. It also keeps the raw export manifest available for cross-checking if the derived split layout changes.

In [ ]:
def load_yaml(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return yaml.safe_load(handle)


def resolve_path(value: str | Path, base_dir: Path) -> Path:
    path = Path(value)
    if not path.is_absolute():
        path = (base_dir / path).resolve()
    return path


def infer_label_dir(image_dir: Path) -> Path:
    parts = list(image_dir.parts)
    if 'images' in parts:
        idx = len(parts) - 1 - parts[::-1].index('images')
        parts[idx] = 'labels'
        return Path(*parts)
    return image_dir.parent.parent / 'labels' / image_dir.name


def normalize_names(raw_names: dict | list) -> dict[int, str]:
    if isinstance(raw_names, list):
        return {idx: str(name) for idx, name in enumerate(raw_names)}
    return {int(idx): str(name) for idx, name in raw_names.items()}


def discover_split_dirs(dataset_yaml: Path, split_order: Iterable[str]) -> tuple[dict, dict[int, str], dict[str, dict[str, Path]]]:
    cfg = load_yaml(dataset_yaml)
    yaml_base = dataset_yaml.parent
    names = normalize_names(cfg.get('names', {}))
    split_dirs: dict[str, dict[str, Path]] = {}
    for split in split_order:
        if split not in cfg:
            continue
        image_dir = resolve_path(cfg[split], yaml_base)
        label_dir = infer_label_dir(image_dir)
        split_dirs[split] = {'image_dir': image_dir, 'label_dir': label_dir}
    return cfg, names, split_dirs


dataset_cfg, class_names, split_dirs = discover_split_dirs(YOLO_DATASET_YAML, SPLIT_ORDER)
manifest = json.loads(EXPORT_MANIFEST.read_text(encoding='utf-8')) if EXPORT_MANIFEST.exists() else {}

split_dir_df = pd.DataFrame(
    [
        {
            'split': split,
            'image_dir': str(paths['image_dir']),
            'image_dir_exists': paths['image_dir'].exists(),
            'label_dir': str(paths['label_dir']),
            'label_dir_exists': paths['label_dir'].exists(),
        }
        for split, paths in split_dirs.items()
    ]
)

display(split_dir_df)
print(f'Classes ({len(class_names)}): {class_names}')
print(f'Manifest keys: {list(manifest)[:20]}')

## Load Images And Instances

The instance table has one row per YOLO annotation. The image table has one row per image, including empty images. Size bins are computed from **training split bbox-area tertiles** and then applied unchanged to `val` and `test`, which makes the split comparison fair.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def iter_image_paths(image_dir: Path) -> list[Path]:
    return sorted(path for path in image_dir.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)


def read_image_size(image_path: Path) -> tuple[int, int]:
    with Image.open(image_path) as image:
        return image.size


def load_split_tables(split: str, image_dir: Path, label_dir: Path, names: dict[int, str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows = []
    instance_rows = []
    for image_index, image_path in enumerate(iter_image_paths(image_dir)):
        width_px, height_px = read_image_size(image_path)
        label_path = label_dir / f'{image_path.stem}.txt'
        image_rows.append(
            {
                'split': split,
                'image_index': image_index,
                'image_id': f'{split}/{image_path.stem}',
                'image_stem': image_path.stem,
                'image_path': str(image_path),
                'label_path': str(label_path),
                'width_px': width_px,
                'height_px': height_px,
                'image_area_px': width_px * height_px,
                'label_exists': label_path.exists(),
            }
        )
        if not label_path.exists():
            continue
        label_text = label_path.read_text(encoding='utf-8').strip()
        if not label_text:
            continue
        for instance_index, line in enumerate(label_text.splitlines()):
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f'Expected 5 YOLO columns in {label_path}, got: {line!r}')
            class_idx = int(parts[0])
            x_center, y_center, bbox_w_norm, bbox_h_norm = map(float, parts[1:])
            bbox_w_px = bbox_w_norm * width_px
            bbox_h_px = bbox_h_norm * height_px
            instance_rows.append(
                {
                    'split': split,
                    'image_index': image_index,
                    'image_id': f'{split}/{image_path.stem}',
                    'image_stem': image_path.stem,
                    'image_path': str(image_path),
                    'label_path': str(label_path),
                    'instance_index': instance_index,
                    'class_idx': class_idx,
                    'class_label': names.get(class_idx, str(class_idx)),
                    'bbox_center_x_norm': x_center,
                    'bbox_center_y_norm': y_center,
                    'bbox_w_norm': bbox_w_norm,
                    'bbox_h_norm': bbox_h_norm,
                    'bbox_area_ratio': bbox_w_norm * bbox_h_norm,
                    'bbox_w_px': bbox_w_px,
                    'bbox_h_px': bbox_h_px,
                    'bbox_area_px': bbox_w_px * bbox_h_px,
                    'width_px': width_px,
                    'height_px': height_px,
                    'image_area_px': width_px * height_px,
                }
            )
    return pd.DataFrame(image_rows), pd.DataFrame(instance_rows)


image_parts = []
instance_parts = []
for split in SPLIT_ORDER:
    paths = split_dirs[split]
    split_image_df, split_instance_df = load_split_tables(split, paths['image_dir'], paths['label_dir'], class_names)
    image_parts.append(split_image_df)
    instance_parts.append(split_instance_df)

image_df = pd.concat(image_parts, ignore_index=True)
instance_df = pd.concat(instance_parts, ignore_index=True)

if instance_df.empty:
    raise ValueError('No annotations were loaded. Check split label directories above.')

train_area = instance_df.loc[instance_df['split'] == 'train', 'bbox_area_ratio'].astype(float)
SIZE_Q33, SIZE_Q67 = train_area.quantile([1.0 / 3.0, 2.0 / 3.0]).tolist()
instance_df = add_position_bin_columns(instance_df)
instance_df['size_bin'] = assign_bins_from_thresholds(instance_df['bbox_area_ratio'], SIZE_Q33, SIZE_Q67)
instance_df['slice_key'] = list(zip(instance_df['class_label'].astype(str), instance_df['size_bin'].astype(str), instance_df['position_bin'].astype(str)))
instance_df['slice_label'] = instance_df['slice_key'].map(lambda item: f'class={item[0]} | size={item[1]} | pos={item[2]}')

per_image_counts = (
    instance_df.groupby(['split', 'image_stem'], observed=True)
    .agg(
        n_instances=('class_idx', 'size'),
        n_classes=('class_idx', 'nunique'),
        n_small_instances=('size_bin', lambda s: int((s.astype(str) == 'small').sum())),
        bbox_area_ratio_mean=('bbox_area_ratio', 'mean'),
        bbox_area_ratio_median=('bbox_area_ratio', 'median'),
    )
    .reset_index()
)
image_df = image_df.merge(per_image_counts, on=['split', 'image_stem'], how='left')
for col in ['n_instances', 'n_classes', 'n_small_instances']:
    image_df[col] = image_df[col].fillna(0).astype(int)
image_df['is_empty'] = image_df['n_instances'].eq(0)
image_df['small_instance_share'] = np.where(image_df['n_instances'] > 0, image_df['n_small_instances'] / image_df['n_instances'], np.nan)

print(f'Loaded {len(image_df):,} images and {len(instance_df):,} instances')
print(f'Train-derived bbox area thresholds: q33={SIZE_Q33:.8f}, q67={SIZE_Q67:.8f}')
display(image_df.groupby('split').size().rename('images').reindex(SPLIT_ORDER).to_frame())
display(instance_df.groupby('split').size().rename('instances').reindex(SPLIT_ORDER).to_frame())

## Headline Split Summary

Use this first table to identify obvious split difficulty differences. Higher validation metrics are expected when `val` has fewer empty images, more/larger objects per image, lower class imbalance, or a slice distribution closer to the training distribution than `test`.

In [ ]:
def gini(values: pd.Series | np.ndarray) -> float:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0 or np.all(arr == 0):
        return 0.0
    arr = np.sort(arr)
    n = arr.size
    return float((2 * np.arange(1, n + 1) @ arr) / (n * arr.sum()) - (n + 1) / n)


def split_class_gini(split: str) -> float:
    counts = instance_df.loc[instance_df['split'] == split, 'class_label'].value_counts().reindex(list(class_names.values()), fill_value=0)
    return gini(counts.to_numpy())


headline_rows = []
for split in SPLIT_ORDER:
    img = image_df.loc[image_df['split'] == split]
    inst = instance_df.loc[instance_df['split'] == split]
    headline_rows.append(
        {
            'split': split,
            'n_images': len(img),
            'n_empty_images': int(img['is_empty'].sum()),
            'empty_image_share': img['is_empty'].mean(),
            'n_instances': len(inst),
            'instances_per_image_mean': img['n_instances'].mean(),
            'instances_per_image_median': img['n_instances'].median(),
            'instances_per_image_p90': img['n_instances'].quantile(0.90),
            'classes_per_image_mean': img['n_classes'].mean(),
            'small_instance_share': (inst['size_bin'].astype(str) == 'small').mean(),
            'bbox_area_ratio_median': inst['bbox_area_ratio'].median(),
            'bbox_area_px_median': inst['bbox_area_px'].median(),
            'class_count_gini': split_class_gini(split),
            'n_present_classes': inst['class_label'].nunique(),
        }
    )
headline_df = pd.DataFrame(headline_rows)
display(headline_df.style.format({
    'empty_image_share': '{:.3f}',
    'instances_per_image_mean': '{:.2f}',
    'instances_per_image_median': '{:.1f}',
    'instances_per_image_p90': '{:.1f}',
    'classes_per_image_mean': '{:.2f}',
    'small_instance_share': '{:.3f}',
    'bbox_area_ratio_median': '{:.6f}',
    'bbox_area_px_median': '{:.1f}',
    'class_count_gini': '{:.3f}',
}))

val_row = headline_df.set_index('split').loc['val'] if 'val' in headline_df['split'].values else None
test_row = headline_df.set_index('split').loc['test'] if 'test' in headline_df['split'].values else None
if val_row is not None and test_row is not None:
    bullets = []
    for col, label, direction in [
        ('empty_image_share', 'empty-image share', 'lower'),
        ('instances_per_image_mean', 'instances per image', 'higher'),
        ('small_instance_share', 'small-object share', 'lower'),
        ('bbox_area_ratio_median', 'median normalized box area', 'higher'),
        ('class_count_gini', 'class imbalance (Gini)', 'lower'),
    ]:
        delta = float(val_row[col] - test_row[col])
        relation = 'higher' if delta > 0 else 'lower' if delta < 0 else 'equal'
        bullets.append(f'- Validation has **{relation} {label}** than test (`val - test = {delta:.4g}`); easier usually means `{direction}` for this metric.')
    display(Markdown('\n'.join(bullets)))

## Single-Instance Analysis

These cells focus on individual object annotations. Look for classes or small/edge-position boxes that are much more common in `test` than in `val`: those are natural candidates for lower test mAP/recall.

In [ ]:
class_counts = (
    instance_df.groupby(['split', 'class_label'], observed=True)
    .size()
    .rename('count')
    .reset_index()
)
class_counts['share'] = class_counts['count'] / class_counts.groupby('split')['count'].transform('sum')
class_pivot = (
    class_counts.pivot(index='class_label', columns='split', values='share')
    .reindex(columns=SPLIT_ORDER)
    .fillna(0.0)
)
if {'val', 'test'}.issubset(class_pivot.columns):
    class_pivot['val_minus_test_share'] = class_pivot['val'] - class_pivot['test']

display(class_pivot.sort_values('val_minus_test_share' if 'val_minus_test_share' in class_pivot else SPLIT_ORDER[0]))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=class_counts, x='class_label', y='count', hue='split', hue_order=SPLIT_ORDER, ax=axes[0])
axes[0].set_title('Instance count by class')
axes[0].tick_params(axis='x', rotation=45)
sns.barplot(data=class_counts, x='class_label', y='share', hue='split', hue_order=SPLIT_ORDER, ax=axes[1])
axes[1].set_title('Instance share by class')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=instance_df, x='split', y='bbox_area_ratio', order=SPLIT_ORDER, showfliers=False, ax=axes[0])
axes[0].set_title('Normalized box area distribution')
axes[0].set_yscale('log')
sns.ecdfplot(data=instance_df, x='bbox_area_ratio', hue='split', hue_order=SPLIT_ORDER, ax=axes[1])
axes[1].set_title('CDF of normalized box area')
axes[1].set_xscale('log')
sns.scatterplot(
    data=instance_df.sample(min(len(instance_df), 15_000), random_state=RANDOM_SEED),
    x='bbox_center_x_norm',
    y='bbox_center_y_norm',
    hue='split',
    hue_order=SPLIT_ORDER,
    alpha=0.35,
    s=10,
    ax=axes[2],
)
axes[2].invert_yaxis()
axes[2].set_title('Object centers by split')
axes[2].set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

size_counts = (
    instance_df.groupby(['split', 'size_bin'], observed=False)
    .size()
    .rename('count')
    .reset_index()
)
size_counts['share'] = size_counts['count'] / size_counts.groupby('split')['count'].transform('sum')
position_counts = (
    instance_df.groupby(['split', 'position_bin'], observed=False)
    .size()
    .rename('count')
    .reset_index()
)
position_counts['share'] = position_counts['count'] / position_counts.groupby('split')['count'].transform('sum')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=size_counts, x='size_bin', y='share', hue='split', hue_order=SPLIT_ORDER, order=list(SIZE_BIN_ORDER), ax=axes[0])
axes[0].set_title('Size-bin share')
sns.barplot(data=position_counts, x='position_bin', y='share', hue='split', hue_order=SPLIT_ORDER, order=list(POSITION_BIN_ORDER), ax=axes[1])
axes[1].set_title('Position-bin share')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Slice Distribution Analysis

A slice is one `(class_label, size_bin, position_bin)` tuple. This view is stricter than class balance alone: a split can have similar class shares but still be harder if it puts many objects into rare small/edge slices that the model saw less often during training.

In [ ]:
def entropy(probs: np.ndarray) -> float:
    probs = probs[probs > 0]
    if probs.size == 0:
        return 0.0
    return float(-(probs * np.log2(probs)).sum())


def js_divergence(left: pd.Series, right: pd.Series) -> float:
    aligned = pd.concat([left.rename('left'), right.rename('right')], axis=1).fillna(0.0).astype(float)
    p = aligned['left'].to_numpy()
    q = aligned['right'].to_numpy()
    p = p / p.sum() if p.sum() else p
    q = q / q.sum() if q.sum() else q
    m = 0.5 * (p + q)
    def kl(a: np.ndarray, b: np.ndarray) -> float:
        mask = a > 0
        return float((a[mask] * np.log2(a[mask] / b[mask])).sum())
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def dense_slice_counts(split: str) -> pd.DataFrame:
    split_instances = instance_df.loc[instance_df['split'] == split].copy()
    table = build_slice_counts_table(split_instances, list(class_names.values()))
    total = max(int(table['count'].sum()), 1)
    table['split'] = split
    table['mass_share'] = table['count'] / total
    table = table.sort_values('count', ascending=False).reset_index(drop=True)
    table['rank'] = np.arange(1, len(table) + 1)
    table['cumulative_mass_share'] = table['mass_share'].cumsum()
    table['slice_label'] = table.apply(lambda row: f"class={row['class_label']} | size={row['size_bin']} | pos={row['position_bin']}", axis=1)
    return table


slice_tables = {split: dense_slice_counts(split) for split in SPLIT_ORDER}
slice_df = pd.concat(slice_tables.values(), ignore_index=True)

slice_summary_rows = []
for split, table in slice_tables.items():
    counts = table['count'].to_numpy(dtype=float)
    probs = counts / counts.sum() if counts.sum() else counts
    n_possible = len(table)
    active = int((counts > 0).sum())
    slice_summary_rows.append(
        {
            'split': split,
            'possible_slices': n_possible,
            'active_slices': active,
            'coverage_fraction': active / n_possible if n_possible else 0.0,
            'normalized_entropy': entropy(probs) / math.log2(n_possible) if n_possible > 1 else 0.0,
            'slice_count_gini': gini(counts),
            'top_15_mass_share': float(table.head(15)['mass_share'].sum()),
            'singleton_slices': int((counts == 1).sum()),
            'empty_slices': int((counts == 0).sum()),
        }
    )
slice_summary_df = pd.DataFrame(slice_summary_rows)

distance_rows = []
for left, right in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    if left in slice_tables and right in slice_tables:
        left_mass = slice_tables[left].set_index(['class_label', 'size_bin', 'position_bin'])['count']
        right_mass = slice_tables[right].set_index(['class_label', 'size_bin', 'position_bin'])['count']
        distance_rows.append({'left': left, 'right': right, 'slice_js_divergence': js_divergence(left_mass, right_mass)})
distance_df = pd.DataFrame(distance_rows)

display(slice_summary_df.style.format({
    'coverage_fraction': '{:.3f}',
    'normalized_entropy': '{:.3f}',
    'slice_count_gini': '{:.3f}',
    'top_15_mass_share': '{:.3f}',
}))
display(distance_df.style.format({'slice_js_divergence': '{:.4f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for split in SPLIT_ORDER:
    table = slice_tables[split]
    axes[0].plot(table['rank'], table['count'], label=split)
    axes[1].plot(table['rank'], table['cumulative_mass_share'], label=split)
axes[0].set_title('Ranked slice counts')
axes[0].set_xlabel('Slice rank')
axes[0].set_ylabel('Count')
axes[0].set_yscale('symlog')
axes[1].set_title('Cumulative slice mass')
axes[1].set_xlabel('Slice rank')
axes[1].set_ylabel('Cumulative share')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

for split in SPLIT_ORDER:
    display(Markdown(f'### Top slices: `{split}`'))
    display(slice_tables[split].head(15)[['class_label', 'size_bin', 'position_bin', 'count', 'mass_share', 'cumulative_mass_share']])

In [ ]:
if {'val', 'test'}.issubset(slice_tables):
    val_mass = slice_tables['val'].set_index(['class_label', 'size_bin', 'position_bin'])['mass_share']
    test_mass = slice_tables['test'].set_index(['class_label', 'size_bin', 'position_bin'])['mass_share']
    val_test_delta = pd.concat([val_mass.rename('val_share'), test_mass.rename('test_share')], axis=1).fillna(0.0)
    val_test_delta['val_minus_test_share'] = val_test_delta['val_share'] - val_test_delta['test_share']
    val_test_delta = val_test_delta.reset_index()
    val_test_delta['abs_delta'] = val_test_delta['val_minus_test_share'].abs()

    display(Markdown('### Slices over-represented in validation'))
    display(val_test_delta.sort_values('val_minus_test_share', ascending=False).head(20))
    display(Markdown('### Slices over-represented in test'))
    display(val_test_delta.sort_values('val_minus_test_share', ascending=True).head(20))

    train_counts = slice_tables['train'].set_index(['class_label', 'size_bin', 'position_bin'])['count']
    hard_test = val_test_delta.copy()
    hard_test['train_count'] = hard_test.set_index(['class_label', 'size_bin', 'position_bin']).index.map(train_counts).astype(int)
    hard_test['candidate_issue'] = np.where(
        (hard_test['test_share'] > hard_test['val_share']) & (hard_test['train_count'] <= 5),
        'test-heavy and rare in train',
        '',
    )
    display(Markdown('### Candidate difficult slices: test-heavy and rare in train'))
    display(hard_test.loc[hard_test['candidate_issue'].ne('')].sort_values(['test_share', 'train_count'], ascending=[False, True]).head(30))

## Image-Level Properties

This section treats each image as the unit of analysis. Higher validation metrics are easier to explain if validation images have more annotated targets, fewer empty frames, more centered objects, fewer tiny objects, or less visual variation.

In [ ]:
image_summary_rows = []
for split in SPLIT_ORDER:
    img = image_df.loc[image_df['split'] == split].copy()
    non_empty = img.loc[~img['is_empty']]
    image_summary_rows.append(
        {
            'split': split,
            'images': len(img),
            'empty_share': img['is_empty'].mean(),
            'instances_per_image_mean': img['n_instances'].mean(),
            'instances_per_non_empty_image_mean': non_empty['n_instances'].mean(),
            'classes_per_image_mean': img['n_classes'].mean(),
            'small_share_per_non_empty_image_mean': non_empty['small_instance_share'].mean(),
            'mean_bbox_area_ratio_per_non_empty_image': non_empty['bbox_area_ratio_mean'].mean(),
            'median_bbox_area_ratio_per_non_empty_image': non_empty['bbox_area_ratio_median'].median(),
            'image_width_median': img['width_px'].median(),
            'image_height_median': img['height_px'].median(),
        }
    )
image_summary_df = pd.DataFrame(image_summary_rows)
display(image_summary_df.style.format({
    'empty_share': '{:.3f}',
    'instances_per_image_mean': '{:.2f}',
    'instances_per_non_empty_image_mean': '{:.2f}',
    'classes_per_image_mean': '{:.2f}',
    'small_share_per_non_empty_image_mean': '{:.3f}',
    'mean_bbox_area_ratio_per_non_empty_image': '{:.6f}',
    'median_bbox_area_ratio_per_non_empty_image': '{:.6f}',
}))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(data=image_df, x='n_instances', hue='split', hue_order=SPLIT_ORDER, discrete=True, stat='density', common_norm=False, ax=axes[0, 0])
axes[0, 0].set_title('Instances per image')
sns.histplot(data=image_df, x='n_classes', hue='split', hue_order=SPLIT_ORDER, discrete=True, stat='density', common_norm=False, ax=axes[0, 1])
axes[0, 1].set_title('Classes per image')
sns.boxplot(data=image_df.loc[~image_df['is_empty']], x='split', y='small_instance_share', order=SPLIT_ORDER, showfliers=False, ax=axes[1, 0])
axes[1, 0].set_title('Small-object share per non-empty image')
sns.boxplot(data=image_df.loc[~image_df['is_empty']], x='split', y='bbox_area_ratio_mean', order=SPLIT_ORDER, showfliers=False, ax=axes[1, 1])
axes[1, 1].set_title('Mean box area per non-empty image')
axes[1, 1].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
def sample_pixel_stats(image_table: pd.DataFrame, per_split: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rows = []
    rng = np.random.default_rng(seed)
    for split, group in image_table.groupby('split', sort=False):
        sample_n = min(per_split, len(group))
        if sample_n == 0:
            continue
        sampled_indices = rng.choice(group.index.to_numpy(), size=sample_n, replace=False)
        for _, row in group.loc[sampled_indices].iterrows():
            path = Path(row['image_path'])
            with Image.open(path) as image:
                arr = np.asarray(image.convert('L'), dtype=np.float32) / 255.0
            rows.append(
                {
                    'split': split,
                    'image_stem': row['image_stem'],
                    'brightness_mean': float(arr.mean()),
                    'brightness_std': float(arr.std()),
                    'brightness_p05': float(np.quantile(arr, 0.05)),
                    'brightness_p95': float(np.quantile(arr, 0.95)),
                    'contrast_p95_minus_p05': float(np.quantile(arr, 0.95) - np.quantile(arr, 0.05)),
                }
            )
    return pd.DataFrame(rows)


pixel_df = sample_pixel_stats(image_df, PIXEL_STATS_SAMPLE_PER_SPLIT)
display(pixel_df.groupby('split').agg(
    sampled_images=('image_stem', 'size'),
    brightness_mean=('brightness_mean', 'mean'),
    brightness_std_mean=('brightness_std', 'mean'),
    contrast_mean=('contrast_p95_minus_p05', 'mean'),
).reindex(SPLIT_ORDER))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(data=pixel_df, x='brightness_mean', hue='split', hue_order=SPLIT_ORDER, common_norm=False, ax=axes[0])
axes[0].set_title('Sampled image brightness')
sns.kdeplot(data=pixel_df, x='contrast_p95_minus_p05', hue='split', hue_order=SPLIT_ORDER, common_norm=False, ax=axes[1])
axes[1].set_title('Sampled image contrast')
plt.tight_layout()
plt.show()

## Why Validation Metrics Can Be Higher

This cell produces a compact, rule-based diagnostic table. Positive `easier_for_val` entries indicate properties where validation is expected to score higher than test for the same checkpoint.

In [ ]:
def value_for_split(table: pd.DataFrame, split: str, metric: str) -> float:
    return float(table.set_index('split').loc[split, metric])


reason_rows = []
if {'val', 'test'}.issubset(set(SPLIT_ORDER)):
    comparisons = [
        {
            'factor': 'empty frames',
            'metric': 'empty_image_share',
            'source': 'headline',
            'val': value_for_split(headline_df, 'val', 'empty_image_share'),
            'test': value_for_split(headline_df, 'test', 'empty_image_share'),
            'easier_when': 'lower',
            'why_it_matters': 'Fewer empty frames usually means fewer pure false-positive opportunities and a target-rich validation set.',
        },
        {
            'factor': 'target density',
            'metric': 'instances_per_image_mean',
            'source': 'headline',
            'val': value_for_split(headline_df, 'val', 'instances_per_image_mean'),
            'test': value_for_split(headline_df, 'test', 'instances_per_image_mean'),
            'easier_when': 'higher',
            'why_it_matters': 'More annotated objects per image can raise recall and stabilize precision/recall estimates.',
        },
        {
            'factor': 'small object burden',
            'metric': 'small_instance_share',
            'source': 'headline',
            'val': value_for_split(headline_df, 'val', 'small_instance_share'),
            'test': value_for_split(headline_df, 'test', 'small_instance_share'),
            'easier_when': 'lower',
            'why_it_matters': 'Tiny thermal targets are harder to localize and hurt mAP50-95 especially strongly.',
        },
        {
            'factor': 'object scale',
            'metric': 'bbox_area_ratio_median',
            'source': 'headline',
            'val': value_for_split(headline_df, 'val', 'bbox_area_ratio_median'),
            'test': value_for_split(headline_df, 'test', 'bbox_area_ratio_median'),
            'easier_when': 'higher',
            'why_it_matters': 'Larger median boxes are easier for YOLO to detect and localize.',
        },
        {
            'factor': 'class imbalance',
            'metric': 'class_count_gini',
            'source': 'headline',
            'val': value_for_split(headline_df, 'val', 'class_count_gini'),
            'test': value_for_split(headline_df, 'test', 'class_count_gini'),
            'easier_when': 'lower',
            'why_it_matters': 'Lower imbalance usually means fewer tail-class misses dominate macro-style metrics.',
        },
        {
            'factor': 'train similarity',
            'metric': 'slice_js_divergence_from_train',
            'source': 'slice distances',
            'val': float(distance_df.query("left == 'train' and right == 'val'")['slice_js_divergence'].iloc[0]),
            'test': float(distance_df.query("left == 'train' and right == 'test'")['slice_js_divergence'].iloc[0]),
            'easier_when': 'lower',
            'why_it_matters': 'A split whose joint class/size/position distribution is closer to train should be easier for the trained checkpoint.',
        },
        {
            'factor': 'slice concentration',
            'metric': 'top_15_mass_share',
            'source': 'slice summary',
            'val': value_for_split(slice_summary_df, 'val', 'top_15_mass_share'),
            'test': value_for_split(slice_summary_df, 'test', 'top_15_mass_share'),
            'easier_when': 'higher',
            'why_it_matters': 'Concentration in dominant slices can inflate performance if those slices are well represented in training.',
        },
    ]
    for row in comparisons:
        delta = row['val'] - row['test']
        if row['easier_when'] == 'lower':
            easier = row['val'] < row['test']
        else:
            easier = row['val'] > row['test']
        row['val_minus_test'] = delta
        row['easier_for_val'] = bool(easier)
        reason_rows.append(row)

reason_df = pd.DataFrame(reason_rows)
display(reason_df[['factor', 'metric', 'val', 'test', 'val_minus_test', 'easier_when', 'easier_for_val', 'why_it_matters']].style.format({
    'val': '{:.6g}',
    'test': '{:.6g}',
    'val_minus_test': '{:.6g}',
}))

if not reason_df.empty:
    hits = reason_df.loc[reason_df['easier_for_val'], 'factor'].tolist()
    misses = reason_df.loc[~reason_df['easier_for_val'], 'factor'].tolist()
    summary = [
        '### Interpretation',
        f'- Validation looks easier than test on **{len(hits)} / {len(reason_df)}** checked factors: {", ".join(hits) if hits else "none"}.',
    ]
    if misses:
        summary.append(f'- Factors that do not support the easier-validation hypothesis: {", ".join(misses)}.')
    summary.append('- Use the over/under-represented slice tables above to connect metric gaps to concrete object types and positions.')
    display(Markdown('\n'.join(summary)))

## Save Analysis Tables

Rerun this cell after changing the dataset path or analysis parameters. The CSVs are useful for comparing notebooks, reports, and experiment summaries without re-executing plots.

In [ ]:
headline_df.to_csv(OUTPUT_DIR / 'split_headline_summary.csv', index=False)
class_pivot.reset_index().to_csv(OUTPUT_DIR / 'class_distribution_comparison.csv', index=False)
slice_summary_df.to_csv(OUTPUT_DIR / 'slice_summary.csv', index=False)
distance_df.to_csv(OUTPUT_DIR / 'slice_distribution_distances.csv', index=False)
image_summary_df.to_csv(OUTPUT_DIR / 'image_level_summary.csv', index=False)
reason_df.to_csv(OUTPUT_DIR / 'why_eval_metrics_are_higher.csv', index=False)
slice_df.to_csv(OUTPUT_DIR / 'dense_slice_counts.csv', index=False)

print(f'Saved analysis tables to: {OUTPUT_DIR}')